# Spark Transformations & Actions

Examples of lazy transformations (narrow and wide) versus actions that actually trigger execution.

In [0]:
## Setup - the employee dataset used by all examples below

from pyspark.sql import functions as F

data = [
    (1, "Ravi", "IT", 50000, "CLOSED", "Hyderabad"),
    (2, "Arjun", "HR", 35000, "OPEN", "Chennai"),
    (3, "Sita", "IT", 65000, "CLOSED", "Hyderabad"),
    (4, "Priya", "Finance", 45000, "OPEN", "Bangalore"),
    (5, "Kiran", "IT", 70000, "CLOSED", "Chennai"),
    (6, "Anu", "HR", 30000, "CLOSED", "Hyderabad"),
    (7, "Manoj", "Finance", 55000, "OPEN", "Bangalore"),
    (8, "Divya", "IT", 80000, "OPEN", "Hyderabad"),
    (9, "Rahul", "HR", 40000, "CLOSED", "Chennai"),
    (10, "Sneha", "Finance", 60000, "OPEN", "Hyderabad")
]
employees = spark.createDataFrame(data, ["id", "name", "department", "salary", "status", "city"])

employees.show()

+---+-----+----------+------+------+---------+
| id| name|department|salary|status|     city|
+---+-----+----------+------+------+---------+
|  1| Ravi|        IT| 50000|CLOSED|Hyderabad|
|  2|Arjun|        HR| 35000|  OPEN|  Chennai|
|  3| Sita|        IT| 65000|CLOSED|Hyderabad|
|  4|Priya|   Finance| 45000|  OPEN|Bangalore|
|  5|Kiran|        IT| 70000|CLOSED|  Chennai|
|  6|  Anu|        HR| 30000|CLOSED|Hyderabad|
|  7|Manoj|   Finance| 55000|  OPEN|Bangalore|
|  8|Divya|        IT| 80000|  OPEN|Hyderabad|
|  9|Rahul|        HR| 40000|CLOSED|  Chennai|
| 10|Sneha|   Finance| 60000|  OPEN|Hyderabad|
+---+-----+----------+------+------+---------+



In [0]:
## NARROW transformations - each row is processed independently, no shuffle

# select / drop / withColumn / filter (where) are all narrow

selected = employees.select("name", "department", "salary")  # pick columns

with_bonus = selected.withColumn("salary_with_bonus", F.col("salary") * 1.10)  # add/derive a column

dropped = with_bonus.drop("salary")  # remove a column

filtered = dropped.filter(F.col("salary_with_bonus") >= 60000)  # keep matching rows (WHERE)

print("Nothing computed yet - transformations are LAZY!")

filtered.show()  # <- the show() ACTION triggers the whole chain above

Nothing computed yet - transformations are LAZY!
+-----+----------+-----------------+
| name|department|salary_with_bonus|
+-----+----------+-----------------+
| Sita|        IT|          71500.0|
|Kiran|        IT|          77000.0|
|Manoj|   Finance|60500.00000000001|
|Divya|        IT|          88000.0|
|Sneha|   Finance|          66000.0|
+-----+----------+-----------------+



In [0]:
## WIDE transformations - require a shuffle (data moves between partitions)

# groupBy + agg: shuffle rows so same department lands together
agg = employees.groupBy("department") \
    .agg(
        F.count("*").alias("emp_count"),
        F.sum("salary").alias("total_salary"),
        F.avg("salary").alias("avg_salary")
    )
agg.show()

# orderBy: a global sort is also a wide transformation (shuffle by sort key)
print("--- orderBy (wide) ---")
employees.orderBy(F.col("salary").desc()).show(3)

# repartition / distinct / join are wide too
print("--- distinct departments (wide) ---")
employees.select("department").distinct().show()

+----------+---------+------------+------------------+
|department|emp_count|total_salary|        avg_salary|
+----------+---------+------------+------------------+
|        IT|        4|      265000|           66250.0|
|        HR|        3|      105000|           35000.0|
|   Finance|        3|      160000|53333.333333333336|
+----------+---------+------------+------------------+

--- orderBy (wide) ---
+---+-----+----------+------+------+---------+
| id| name|department|salary|status|     city|
+---+-----+----------+------+------+---------+
|  8|Divya|        IT| 80000|  OPEN|Hyderabad|
|  5|Kiran|        IT| 70000|CLOSED|  Chennai|
|  3| Sita|        IT| 65000|CLOSED|Hyderabad|
+---+-----+----------+------+------+---------+
only showing top 3 rows
--- distinct departments (wide) ---
+----------+
|department|
+----------+
|        IT|
|        HR|
|   Finance|
+----------+



In [0]:
## Laziness demo - transformations only build a PLAN, actions EXECUTE it

result = employees \
    .filter(F.col("status") == "OPEN") \
    .withColumn("dept_upper", F.upper("department")) \
    .select("name", "dept_upper", "salary")

print("Chain of 3 transformations defined - nothing has run yet.")

# Look at the LOGICAL PLAN that Spark built (still no data processed):
print("\n=== Logical plan ===")
result.explain()

# Only an action executes the plan:
print("\n=== count() action fires now ===")
print("OPEN employees:", result.count())

Chain of 3 transformations defined - nothing has run yet.

=== Logical plan ===
== Physical Plan ==
LocalTableScan [name#10617, dept_upper#10623, salary#10619L]


== Photon Explanation ==
Photon does not fully support the query because:
		Unsupported node: LocalTableScan [name#10617, dept_upper#10623, salary#10619L].

Reference node:
	LocalTableScan [name#10617, dept_upper#10623, salary#10619L]


=== count() action fires now ===
OPEN employees: 5


In [0]:
## ACTIONS - they trigger execution and RETURN something

# show()    -> prints rows to the console (nothing returned)
employees.show(3)

# count()   -> returns the number of rows
print("count()      :", employees.count())

# take(n)   -> returns the first n rows as a local list
print("take(2)      :", employees.take(2))

# first()   -> returns a single Row
print("first()      :", employees.first())

# collect()  -> returns ALL rows to the driver (careful with big data!)
all_rows = employees.collect()
print("collect()    : returned", len(all_rows), "rows to the driver")

# printSchema() / columns are also actions on the metadata
employees.printSchema()

+---+-----+----------+------+------+---------+
| id| name|department|salary|status|     city|
+---+-----+----------+------+------+---------+
|  1| Ravi|        IT| 50000|CLOSED|Hyderabad|
|  2|Arjun|        HR| 35000|  OPEN|  Chennai|
|  3| Sita|        IT| 65000|CLOSED|Hyderabad|
+---+-----+----------+------+------+---------+
only showing top 3 rows
count()      : 10
take(2)      : [Row(id=1, name='Ravi', department='IT', salary=50000, status='CLOSED', city='Hyderabad'), Row(id=2, name='Arjun', department='HR', salary=35000, status='OPEN', city='Chennai')]
first()      : Row(id=1, name='Ravi', department='IT', salary=50000, status='CLOSED', city='Hyderabad')
collect()    : returned 10 rows to the driver
root
 |-- id: long (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- status: string (nullable = true)
 |-- city: string (nullable = true)



In [0]:
## Action with a real side effect - write() persists data

# write() is an action: the lazy plan finally executes and the data is saved.
# The "noop" sink EXECUTES the full pipeline but writes NOTHING -
# perfect for demos and benchmarks without creating or overwriting tables.
# In real use you would replace it with e.g. .saveAsTable("my_schema.my_table")
(employees.filter(F.col("status") == "OPEN")
    .write
    .format("noop")
    .mode("overwrite")
    .save()
)

print("write action completed - the OPEN filter pipeline fully executed")

# Prove it: check the results of the same filter without persisting anything
employees.filter(F.col("status") == "OPEN").show()

write action completed - the OPEN filter pipeline fully executed
+---+-----+----------+------+------+---------+
| id| name|department|salary|status|     city|
+---+-----+----------+------+------+---------+
|  2|Arjun|        HR| 35000|  OPEN|  Chennai|
|  4|Priya|   Finance| 45000|  OPEN|Bangalore|
|  7|Manoj|   Finance| 55000|  OPEN|Bangalore|
|  8|Divya|        IT| 80000|  OPEN|Hyderabad|
| 10|Sneha|   Finance| 60000|  OPEN|Hyderabad|
+---+-----+----------+------+------+---------+



# Transformations vs Actions — cheat sheet

| | Transformation | Action |
|---|---|---|
| Returns | A new DataFrame (lazy plan) | A value / side effect |
| Executes immediately? | No — only builds the plan | Yes — triggers the whole chain |
| Examples | `select`, `filter`, `withColumn`, `groupBy`, `join`, `orderBy`, `repartition` | `show`, `count`, `collect`, `take`, `first`, `write`, `printSchema` |

**Narrow vs wide transformations**

* Narrow — no shuffle, each partition maps independently: `select`, `filter`, `withColumn`, `drop`
* Wide — shuffle, rows move across partitions: `groupBy`, `orderBy`, `join`, `distinct`, `repartition`